# Lab 5 — CNNs and ViTs vs. the Curse of Dimensionality
**CS6524: Computer Vision — Immersive Software Engineering, University of Limerick**

**Objective.** Compare a CNN family (**ConvNeXt**) and a Vision Transformer family (**Swin Transformer**) on **monocular depth estimation** (NYU Depth v2), trained **from scratch** on the full training set and on a 6× smaller one, to see which architecture is more robust when data is scarce relative to model capacity.

All reusable code (data loading, model definition, training loop, plotting) lives in the `mypylibs/` package of this repository, so this notebook only shows the experiment itself.

## 0. Environment
Run on a Colab **GPU runtime** (*Runtime → Change runtime type → T4 GPU*). Outputs (checkpoints, loss histories, test scores) are written to Google Drive, and every run resumes from its last epoch if Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/DaraHeaphy/lab5.git'          # this repository
DATA_DIR = '/content/drive/MyDrive/NYU_Depth'                # shortcut to the shared NYU Drive folder
OUT_DIR  = '/content/drive/MyDrive/cs6524_lab5'              # checkpoints, histories, results

import os
if not os.path.exists('/content/lab5'):
    !git clone -q {REPO_URL} /content/lab5
%cd /content/lab5
!nvidia-smi -L

In [ ]:
import torch, pandas as pd
from mypylibs.data import load_nyu, make_splits, NYUDepth, IMG_SIZE
from mypylibs.models import DepthNet, count_params, BACKBONES, FAMILY
from mypylibs.train import TrainConfig, run_experiment, load_model, run_name
from mypylibs import plots

device = 'cuda'
torch.backends.cudnn.benchmark = True

## 1. Experimental setup

### Task
Monocular depth estimation: given a single RGB image, predict a dense per-pixel depth map in metres.

### Data
- **NYU Depth v2** (indoor scenes captured with a Kinect), loaded from the shared Drive folder.
- **Preprocessing.** Every image is resized to **192 × 256** (H × W). This keeps NYU's 3:4 aspect ratio and is divisible by 32, the largest stride of both backbones. RGB uses bilinear, anti-aliased resizing and is normalised with ImageNet statistics. Depth uses **nearest-neighbour** resizing, so we never invent depths by blending a foreground and a background pixel, and it stays in metres. Pixels with depth 0 (missing, or padding created by augmentation) are masked out of both the loss and the metric. The resized set is cached on Drive.
- **Splits** (fixed seed, saved to `splits.json`, identical for every model): **60% train / 20% validation / 20% test**.
- **Reduced training set:** **10% of the whole dataset**, sampled once from the training split. The same images are used by every model. It is 6× smaller than the full training set, while validation and test stay unchanged.

### Models — 4 backbones × 2 training sets = 8 runs
Every backbone is a torchvision implementation with `weights=None`, so it is trained **from scratch**. Its classification head is replaced by the **same lightweight FPN decoder** for every model:
1. Take the 4 stage outputs (strides 4, 8, 16, 32).
2. Project each with a 1×1 conv to 128 channels.
3. Fuse them top-down with upsampling and 3×3 conv–GroupNorm–GELU blocks.
4. Predict 1 channel at 1/4 resolution, upsample bilinearly to full size, and apply a softplus so depth is positive.

Only the encoder changes between runs, so differences can be attributed to the architecture.

| Backbone | Family | Inductive bias |
|---|---|---|
| ConvNeXt-T / ConvNeXt-B | CNN | locality + weight sharing (translation equivariance) baked in |
| Swin-T / Swin-B | ViT | self-attention within shifted local windows; spatial structure must be largely *learned* |

The *tiny* and *base* variants of the two families are matched in size (≈28 M and ≈88 M parameters), which makes the CNN-vs-ViT comparison fair (see the table below).

### Training (identical for all runs)
- **Loss:** Huber (δ = 1 m) on valid pixels.
- **Optimiser:** AdamW (lr 3e-4, weight decay 0.05), 3-epoch linear warm-up, then cosine decay.
- **Batches and epochs:** batch size 16, 50 epochs.
- **Numerics:** mixed precision (fp16), gradient clipping at 1.0, seed 0.
- **Model selection:** the weights with the **lowest validation loss** are kept.
- **Evaluation:** **Mean Absolute Relative Error**, MARE = mean(|d̂ − d| / d) over valid pixels. It is averaged per image, then over the test set.

Using the same number of *epochs* for both training sets (as the brief requires) means the reduced runs also take 6× fewer optimisation steps. We return to this in the discussion.

In [ ]:
BACKBONES_ = list(BACKBONES)            # ['convnext_tiny', 'convnext_base', 'swin_t', 'swin_b']
SUBSETS = ['full', 'reduced']
cfg = TrainConfig(epochs=50, batch_size=16, lr=3e-4, weight_decay=0.05, warmup_epochs=3, huber_delta=1.0)

pd.DataFrame([{'model': b, 'family': FAMILY[b], 'params (M)': round(count_params(DepthNet(b)), 1)}
              for b in BACKBONES_]).set_index('model')

## 2. Data preparation

In [ ]:
rgb, depth = load_nyu(DATA_DIR, cache_path=f'{OUT_DIR}/nyu_{IMG_SIZE[0]}x{IMG_SIZE[1]}.pt')
splits = make_splits(len(rgb), seed=42, path=f'{OUT_DIR}/splits.json')

print(f'{len(rgb)} images, RGB {tuple(rgb.shape[1:])}, depth range {depth[depth > 0].min():.2f}-{depth.max():.2f} m')
pd.Series({k: len(v) for k, v in splits.items()}, name='images').to_frame().T

In [ ]:
def datasets(subset, augment=False):
    train_idx = splits['train'] if subset == 'full' else splits['reduced']
    return (NYUDepth(rgb, depth, train_idx, augment=augment),
            NYUDepth(rgb, depth, splits['val']),
            NYUDepth(rgb, depth, splits['test']))

plots.show_samples(datasets('full')[0])

## 3. Training
Eight runs. Each run is skipped if it already finished, or resumed from its last checkpoint, so re-running this cell after a disconnect is safe. Test MARE is computed once, using the best-on-validation weights.

In [ ]:
histories, results = {}, {}
for subset in SUBSETS:
    for b in BACKBONES_:
        name = run_name(b, subset, augment=False)
        histories[name], results[name] = run_experiment(b, subset, False, datasets, cfg, OUT_DIR, device)

## 4. Training and validation loss curves
Rows: training set. Columns: backbone. The dotted line marks the epoch with the best validation loss (the weights used for testing).

In [ ]:
plots.plot_loss_curves(histories, BACKBONES_, SUBSETS)

## 5. Test-set performance (MARE)
*Degradation* is the relative increase in test MARE when the model is trained on the reduced set instead of the full set. It is our measure of robustness to data scarcity. *Train-val gap* is the validation loss minus the training loss at the selected epoch, a direct measure of overfitting.

In [ ]:
table = plots.results_table(results, BACKBONES_, SUBSETS)
for s in SUBSETS:
    h = {b: histories[f'{b}_{s}'] for b in BACKBONES_}
    table[f'train-val gap ({s})'] = [h[b]['val_loss'][h[b]['best_epoch']] - h[b]['train_loss'][h[b]['best_epoch']]
                                     for b in BACKBONES_]
table.style.format(precision=3)

In [ ]:
plots.plot_mare_bars(table)

In [ ]:
test_ds = datasets('full')[2]
models = {run_name(b, s, False): load_model(b, f'{OUT_DIR}/{run_name(b, s, False)}', device)
          for s in SUBSETS for b in ('convnext_tiny', 'swin_t')}
plots.show_predictions(models, test_ds, indices=[0, 15, 30, 45], device=device)

## 6. Discussion

> ✏️ **Fill in after the runs finish, using the numbers above.** Each bullet says what to look at.

### Accuracy
- Which model has the lowest MARE on the full set? Do CNNs or ViTs win at equal size (ConvNeXt-T vs Swin-T, ConvNeXt-B vs Swin-B)?
- Does the larger (base) variant beat the tiny one, or is the extra capacity wasted or harmful when trained from scratch on ~1k images?

### Robustness (full → reduced)
- Compare the *degradation (%)* column. A smaller degradation means more robust to data scarcity.
- Expected pattern: Swin degrades more than ConvNeXt. Convolutions hard-code locality and translation equivariance, so they need fewer samples to learn useful local filters. Self-attention has to *learn* those spatial priors from data. Check whether your numbers confirm this.

### Generalisation
- Use the loss curves. Look at the gap between train and validation loss, and at whether validation loss stops improving (or rises) while training loss keeps falling, i.e. overfitting.
- Which runs overfit earliest (look at the best epoch)? Which never converge at all (both curves flat and high)?
- Look at the qualitative predictions. Are the reduced-set models just predicting a blurry "average room"?

### The curse of dimensionality
**How is it evident in our experiments?**

The input space is very high-dimensional: 192×256×3 ≈ 147k dimensions. The output is also dense (49k depth values), and the models have 28–88M free parameters. Yet we have only ~870 (full) or ~145 (reduced) training images. The space is covered so sparsely that the models can memorise the training images instead of learning the underlying geometry. Evidence:
- *(fill in)* the train–validation gap, especially on the reduced set and for the base models;
- *(fill in)* the worse test MARE of the base models compared with the tiny ones;
- *(fill in)* the larger degradation of the ViTs, which have weaker priors and so effectively a larger hypothesis space to search with the same data.

**Was it solved?**
*(fill in — most likely: no. Training from scratch, no model reaches a MARE that is useful in practice, and every model degrades when data is reduced.)*

**Next steps to solve it.** Any of these either adds data or shrinks the space the model has to search:
1. **Transfer learning:** start from ImageNet-pretrained weights, the single biggest lever, especially for ViTs.
2. **Data augmentation:** flips, crops, colour jitter (see the bonus below).
3. **More data:** the raw NYU set has ~400k frames from video, and other datasets could be added.
4. **Stronger priors and regularisation:** smaller models, stochastic depth, dropout, early stopping, hybrid CNN–ViT stems.
5. **Self-supervised pre-training:** e.g. MAE / DINO on unlabelled indoor images.

## 7. Bonus — data augmentation
The same 8 runs again, with **joint** augmentations applied to the RGB image and its depth map:
- horizontal flip (p = 0.5)
- random crop (75–100% of the side), resized back to 192 × 256
- rotation (±5°), where the padded corners get depth 0 and are masked
- colour jitter (brightness, contrast, saturation, hue), applied to the RGB image only

Validation and test sets are never augmented. Set `RUN_AUG = True` to run it. To split the work with a lab partner, give each person a subset of `BACKBONES_` and copy the run folders into the same `OUT_DIR`.

In [ ]:
RUN_AUG = False
if RUN_AUG:
    for subset in SUBSETS:
        for b in BACKBONES_:
            name = run_name(b, subset, augment=True)
            histories[name], results[name] = run_experiment(b, subset, True, datasets, cfg, OUT_DIR, device)

    plots.plot_loss_curves(histories, BACKBONES_, SUBSETS, suffix='_aug')
    aug = plots.results_table(results, BACKBONES_, SUBSETS, suffix='_aug')
    comparison = table[['family', 'MARE full', 'MARE reduced', 'degradation (%)']].join(
        aug.drop(columns='family').add_suffix(' (aug)'))
    display(comparison.style.format(precision=3))

> ✏️ **Discuss:** does augmentation reduce test MARE and the full → reduced degradation? Does it help Swin more than ConvNeXt (it is an injected prior, which is exactly what ViTs lack)?